In [ ]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from utils.featureExtractor import featureExtractor
from utils.impChi import Chi2TextFeatureSelector
from sklearn.metrics import classification_report, f1_score

from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier


In [2]:
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

In [3]:
df_train, df_test = train_test_split(train, test_size=0.25, stratify=train['LABEL'], random_state=42)
X_train, y_train = df_train['TEXT'], df_train['LABEL']
X_test, y_test = df_test['TEXT'], df_test['LABEL']

In [4]:
X_train = pd.DataFrame(X_train)

In [5]:
pipeline = Pipeline(
     [
          
          ('feature_ext', featureExtractor()),
          ('impChi', Chi2TextFeatureSelector(k_per_label=45, min_df=3, ngram_range=(1, 3))),
          ('ss', StandardScaler())
     ]
)
X_train_pp = pipeline.fit_transform(pd.DataFrame(X_train), y_train)
X_test_pp = pipeline.transform(pd.DataFrame(X_test))

Fitting TF-IDF (min_df=3, ngrams=(1, 3))...
Selecting top 45 features per label via Chi-Square test...
Total unique TF-IDF features selected: 257


## Hyperparameter tuning
I have used optuna (Bayesian optimization) to search for the most promising models, which are reported below.
I did run it for 50 trials.
### Configurations found
In this way, I have found the following 3 best configurations.
In the end, I will be using only two of them. 
__params_26__ seems to have been lucky.

In [ ]:
params_31 = {'objective':'multiclass',
    'num_class':6,
    'class_weight':'balanced',
    'boosting_type':'gbdt',
    'random_state':42,
    'n_jobs':-1,
    'verbosity':-1,'n_estimators': 730, 'learning_rate': 0.010407498940906855, 'max_depth': 5, 'num_leaves': 54, 'min_child_samples': 37, 'min_split_gain': 0.06963977833346169, 'reg_alpha': 0.21148540123122064, 'reg_lambda': 0.030761881469967566, 'colsample_bytree': 0.3518507408719965, 'subsample': 0.9676247445281039, 'subsample_freq': 6}

params_26 = {'objective':'multiclass',
    'num_class':6,
    'class_weight':'balanced',
    'boosting_type':'gbdt',
    'random_state':42,
    'n_jobs':-1,
    'verbosity':-1,'n_estimators': 400, 'learning_rate': 0.02349778714378209, 'max_depth': 12, 'num_leaves': 248, 'min_child_samples': 67, 'min_split_gain': 0.044000000000000004, 'reg_alpha': 0.2150868646977478, 'reg_lambda': 1.2045035383792522, 'colsample_bytree': 0.3370000000000001, 'subsample': 0.8870000000000001, 'subsample_freq': 4}

params_8 = {'objective':'multiclass',
    'num_class':6,
    'class_weight':'balanced',
    'boosting_type':'gbdt',
    'random_state':42,
    'n_jobs':-1,
    'verbosity':-1,'n_estimators': 922, 'learning_rate': 0.00531016167886792, 'max_depth': 8, 'num_leaves': 50, 'min_child_samples': 46, 'min_split_gain': 0.09890451535072747, 'reg_alpha': 0.013804574251952207, 'reg_lambda': 0.2663921229953439, 'colsample_bytree': 0.24448771356145826, 'subsample': 0.8606974769886846, 'subsample_freq': 5}

In [ ]:
# Training the two best promising models
print('training model 1(trial 31)')
model_31 = LGBMClassifier(**params_31).fit(X_train_pp, y_train)

print("training model 2 (trial 8)")
model_8 = LGBMClassifier(**params_8).fit(X_train_pp, y_train)

# extracting probabilities
probs_31 = model_31.predict_proba(X_test_pp)
probs_8  = model_8.predict_proba(X_test_pp)

blended_probs = (probs_31 + probs_8 ) / 2.0

final_predictions = np.argmax(blended_probs, axis=1)

print(f"\nF1 Macro Score using soft voting: {f1_score(y_test, final_predictions, average='macro'):.4f}")
print(classification_report(y_test, final_predictions, digits=4))

training model 1(trial 31)
training model 2 (trial 8)

F1 Macro Score using soft voting: 0.9548
              precision    recall  f1-score   support

           0     1.0000    0.9974    0.9987       380
           1     0.7600    0.9500    0.8444        20
           2     0.9714    0.8500    0.9067        40
           3     1.0000    1.0000    1.0000        20
           4     0.9836    1.0000    0.9917        60
           5     0.9875    0.9875    0.9875        80

    accuracy                         0.9850       600
   macro avg     0.9504    0.9641    0.9548       600
weighted avg     0.9868    0.9850    0.9853       600



c:\Users\gianb\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\gianb\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [8]:
estimators = [
    ('lgbm_31', CalibratedClassifierCV(
        LGBMClassifier(**params_31),
        method='isotonic', cv=5
    )),
    ('lgbm_8', CalibratedClassifierCV(
        LGBMClassifier(**params_8),
        method='isotonic', cv=5
    )),
]

meta = LogisticRegression(
    class_weight='balanced',
    C=0.1,
    max_iter=1000,
    random_state=42
)

stacking_clf = StackingClassifier(
    estimators=estimators,
    final_estimator=meta,
    cv=5,
    stack_method='predict_proba',
    passthrough=False, # do not let logistic regression to see features.
    n_jobs=-1
)

stacking_clf.fit(X_train_pp, y_train)
y_pred = stacking_clf.predict(X_test_pp)
print(f"Stacking F1 Macro: {f1_score(y_test, y_pred, average='macro'):.4f}")
print(classification_report(y_test, y_pred, digits=4))

c:\Users\gianb\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\gianb\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\gianb\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\gianb\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\gianb\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: U

Stacking F1 Macro: 0.9628
              precision    recall  f1-score   support

           0     1.0000    0.9974    0.9987       380
           1     0.8261    0.9500    0.8837        20
           2     0.9722    0.8750    0.9211        40
           3     1.0000    1.0000    1.0000        20
           4     0.9836    1.0000    0.9917        60
           5     0.9753    0.9875    0.9814        80

    accuracy                         0.9867       600
   macro avg     0.9595    0.9683    0.9628       600
weighted avg     0.9874    0.9867    0.9867       600



In [9]:
full_pipeline = Pipeline(
     [
          ('preprocessing_ext', pipeline),
          ('model', stacking_clf)
     ]
)

X,y = train['TEXT'], train['LABEL']
full_pipeline.fit(pd.DataFrame(X), y)
test_preds = full_pipeline.predict(pd.DataFrame(test['TEXT']))
submission = pd.DataFrame({'ID': np.arange(test.shape[0]), 'LABEL': test_preds})
submission.to_csv('submission.csv', index=False)

Fitting TF-IDF (min_df=3, ngrams=(1, 3))...
Selecting top 45 features per label via Chi-Square test...
Total unique TF-IDF features selected: 256


c:\Users\gianb\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\gianb\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\gianb\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\gianb\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\gianb\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: U